In [ ]:
# pip 安装crewai langchain-openai

import os
from crewai import Agent, Task, Crew
from crewai.tools import tool
import logging

# --- 最佳实践：配置日志记录 ---
# 基本的日志记录设置有助于调试和跟踪机组人员的执行情况。
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- 设置您的 API 密钥 ---
# 对于生产，建议使用更安全的密钥管理方法
# 就像运行时加载的环境变量或秘密管理器一样。
#
# 为您选择的 LLM 提供商设置环境变量（例如 OPENAI_API_KEY）
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
# os.environ["OPENAI_MODEL_NAME"] = "gpt-4o"


# --- 1.重构工具：返回干净的数据 ---
# 该工具现在返回原始数据（浮点数）或引发标准 Python 错误。
# 这使得它更具可重用性，并迫使代理正确处理结果。
@tool("Stock Price Lookup Tool")
def get_stock_price(ticker: str) -> float:
    """
    Fetches the latest simulated stock price for a given stock ticker symbol.
    Returns the price as a float. Raises a ValueError if the ticker is not found.
    """
    logging.info(f"Tool Call: get_stock_price for ticker '{ticker}'")
    simulated_prices = {
        "AAPL": 178.15,
        "GOOGL": 1750.30,
        "MSFT": 425.50,
    }
    price = simulated_prices.get(ticker.upper())

    if price is not None:
        return price
    else:
        # 引发特定错误比返回字符串更好。
        # 代理有能力处理异常并可以决定下一步行动。
        raise ValueError(f"Simulated price for ticker '{ticker.upper()}' not found.")


# --- 2. 定义代理 ---
# 代理定义保持不变，但现在将利用改进的工具。
financial_analyst_agent = Agent(
  role='Senior Financial Analyst',
  goal='Analyze stock data using provided tools and report key prices.',
  backstory="You are an experienced financial analyst adept at using data sources to find stock information. You provide clear, direct answers.",
  verbose=True,
  tools=[get_stock_price],
  # 允许委派可能很有用，但对于这个简单的任务来说并不是必需的。
  allow_delegation=False,
)

# --- 3. 细化任务：更清晰的指令和错误处理 ---
# 任务描述更加具体，指导代理如何做出反应
# 成功的数据检索和潜在的错误。
analyze_aapl_task = Task(
  description=(
      "What is the current simulated stock price for Apple (ticker: AAPL)? "
      "Use the 'Stock Price Lookup Tool' to find it. "
      "If the ticker is not found, you must report that you were unable to retrieve the price."
  ),
  expected_output=(
      "A single, clear sentence stating the simulated stock price for AAPL. "
      "For example: 'The simulated stock price for AAPL is $178.15.' "
      "If the price cannot be found, state that clearly."
  ),
  agent=financial_analyst_agent,
)

# --- 4. 制定人员队伍 ---
# 工作人员协调代理和任务如何协同工作。
financial_crew = Crew(
  agents=[financial_analyst_agent],
  tasks=[analyze_aapl_task],
  verbose=True # 对于生产中不太详细的日志设置为 False
)

# --- 5. 在主执行块中运行 Crew ---
# 使用 __name__ == "__main__": 块是标准的 Python 最佳实践。
def main():
    """Main function to run the crew."""
    # 在开始之前检查 API 密钥以避免运行时错误。
    if not os.environ.get("OPENAI_API_KEY"):
        print("ERROR: The OPENAI_API_KEY environment variable is not set.")
        print("Please set it before running the script.")
        return

    print("\n## Starting the Financial Crew...")
    print("---------------------------------")

    # kickoff 方法开始执行。
    result = financial_crew.kickoff()

    print("\n---------------------------------")
    print("## Crew execution finished.")
    print("\nFinal Result:\n", result)

if __name__ == "__main__":
    main()